# Confirmed strong-lens systems observed in Rubin DP1

This notebook is designed for the authenticated Rubin Science Platform at `data.lsst.cloud`. Data Preview 1 (DP1) contains real LSST Commissioning Camera observations of seven fields obtained from 24 October through 11 December 2024. DP1 access is restricted to Rubin data-rights holders.

SIMBAD is queried for objects whose primary classifications are `gLS` (gravitational-lens system) or `gLe` (gravitational lens). Candidate classifications are excluded. Every catalog row near a DP1 field is checked against the exact DP1 Simple Image Access footprint. The result is therefore **all DP1 overlaps represented by these confirmed SIMBAD classifications on the retrieval date**, not every lens that exists in the images. SIMBAD is curated but incomplete, and DP1 does not provide an authoritative strong-lens catalog.

For each overlap, the notebook retrieves one deep coadd per available band, plots the complete retrieved lens-band sample as a collage, and applies a deliberately simple PCAT model to every usable cutout. Duplicate coadd patches that cover the same target in the same band are represented once. The model uses a circular isothermal deflector and one smooth Gaussian source. It does not represent a publication-quality analysis of complex observed lenses.

In [ ]:
from datetime import date
from io import BytesIO
from pathlib import Path
import math
import re

import astropy.units as u
from astropy.coordinates import SkyCoord
from astropy.io import fits
from astropy.visualization import simple_norm
import matplotlib.pyplot as plt
import numpy as np
import pyvo

try:
    from lsst.afw.fits import MemFileManager
    from lsst.afw.image import MaskedImageF
    from lsst.rsp.service import get_siav2_service
    from lsst.rsp.utils import get_pyvo_auth
except ImportError as error:
    raise RuntimeError(
        'Run this notebook in the Rubin Science Platform DP1 environment. '
        'Local fallback data are intentionally not substituted.'
    ) from error

import pcat
from pcat.roman_lens import RomanLensConfig, gaussian_lens_log_likelihood, render_lens_counts

output_root = Path(pcat.__file__).resolve().parents[1] / 'examples/rubin_dp1_confirmed_strong_lenses'
visual_root = output_root / 'visuals'
print(f'Writing to {visual_root}...')
visual_root.mkdir(parents=True, exist_ok=True)
retrieval_date = date.today().isoformat()
print(f'Catalog and DP1 footprint retrieval date: {retrieval_date}')

## Retrieve the catalog and discover DP1 overlaps

The seven field centers and a generous 1.3 degree prefilter reduce the number of authenticated SIA requests. The subsequent SIA query determines exact image coverage, so the prefilter does not define the final sample. Increase `field_prefilter_radius_deg` if future DP1 documentation reports a larger footprint.

In [ ]:
simbad = pyvo.dal.TAPService('https://simbad.cds.unistra.fr/simbad/sim-tap')
simbad_query = """
SELECT main_id, ra, dec, otype
FROM basic
WHERE otype IN ('gLS', 'gLe')
  AND ra IS NOT NULL AND dec IS NOT NULL
"""
print('Reading from SIMBAD TAP...')
lens_catalog = simbad.search(simbad_query).to_table()
print(f'Retrieved {len(lens_catalog)} confirmed lens-system or lens rows from SIMBAD.')

field_centers = SkyCoord(
    ra=[6.02, 37.86, 40.00, 53.13, 59.10, 95.00, 106.23] * u.deg,
    dec=[-72.08, 6.98, -34.45, -28.10, -48.73, -25.00, -10.51] * u.deg,
)
field_prefilter_radius_deg = 1.3  # [deg]
lens_coordinates = SkyCoord(lens_catalog['ra'], lens_catalog['dec'], unit='deg')
near_field = np.any(
    lens_coordinates[:, None].separation(field_centers[None, :]).deg < field_prefilter_radius_deg,
    axis=1,
)
nearby_lenses = lens_catalog[near_field]
print(f'{len(nearby_lenses)} confirmed catalog rows are near a DP1 field center.')

sia = get_siav2_service('dp1')
preferred_bands = ('i', 'r', 'z', 'g', 'y', 'u')
overlaps = []
for row in nearby_lenses:
    ra_deg = float(row['ra'])  # [deg]
    dec_deg = float(row['dec'])  # [deg]
    results = sia.search(
        pos=(ra_deg, dec_deg, 0.0003),  # [deg]
        calib_level=3,
        dpsubtype='lsst.deep_coadd',
    ).to_table()
    for band in preferred_bands:
        selected = results[np.asarray(results['lsst_band']) == band]
        if len(selected):
            overlaps.append({'catalog': row, 'image': selected[0], 'band': band})

unique_systems = {str(overlap['catalog']['main_id']) for overlap in overlaps}
print(f'Exact DP1 matches: {len(unique_systems)} lens systems in {len(overlaps)} lens-band cutouts.')
for overlap in overlaps:
    row = overlap['catalog']
    print(str(row['main_id']), float(row['ra']), float(row['dec']), overlap['band'])
if not overlaps:
    raise RuntimeError(
        'No confirmed SIMBAD gLS/gLe systems overlap the current DP1 deep-coadd footprint. '
        'An empty result is scientifically valid and must not be replaced with simulated lenses.'
    )

## Retrieve calibrated cutouts and animate the complete matched sample

One deep coadd is retrieved for every available band of every matched catalog system. Duplicate overlapping patches in the same band are represented once. The science plane is in nJy and the variance plane is in nJy². The collage contains every unique lens-band match, and the GIF advances through those same cutouts one frame at a time.

In [ ]:
from pyvo.dal.adhoc import DatalinkResults, SodaQuery
from IPython.display import Image, display
from pcat.plotting import make_image_sequence_animation


def retrieve_cutout(overlap, radius_arcsec=8.0):
    """Return DP1 image and variance cutouts from the authenticated SODA service."""
    row = overlap['catalog']
    session = get_pyvo_auth()
    datalink = DatalinkResults.from_result_url(overlap['image']['access_url'], session=session)
    query = SodaQuery.from_resource(
        datalink,
        datalink.get_adhocservice_by_id('cutout-sync-maskedimage'),
        session=session,
    )
    query.circle = (
        float(row['ra']) * u.deg,
        float(row['dec']) * u.deg,
        radius_arcsec * u.arcsec,
    )
    payload = query.execute_stream().read()
    query.raise_if_error()
    memory = MemFileManager(len(payload))
    memory.setData(payload, len(payload))
    masked_image = MaskedImageF(memory)
    return {
        **overlap,
        'data': np.array(masked_image.image.array, dtype=float, copy=True),  # [nJy pixel^-1]
        'variance': np.array(masked_image.variance.array, dtype=float, copy=True),  # [nJy^2 pixel^-2]
        'mask': np.array(masked_image.mask.array, copy=True),
    }


cutouts = [retrieve_cutout(overlap) for overlap in overlaps]
columns = min(4, len(cutouts))
rows = math.ceil(len(cutouts) / columns)
figure, axes = plt.subplots(rows, columns, figsize=(3.4 * columns, 3.4 * rows), squeeze=False)
for axis in axes.flat:
    axis.set_visible(False)
for axis, cutout in zip(axes.flat, cutouts):
    data = cutout['data']
    norm = simple_norm(data, stretch='asinh', percent=99.5)
    axis.imshow(data, origin='lower', cmap='Greys', norm=norm, interpolation='nearest')
    axis.set_title(f"{str(cutout['catalog']['main_id'])}\n{cutout['band']}-band")
    axis.set_xlabel('Pixel')
    axis.set_ylabel('Pixel')
    axis.grid(False)
    axis.set_visible(True)
figure.tight_layout()
collage_path = visual_root / 'rubin_dp1_confirmed_lens_collage.png'
print(f'Writing to {collage_path}...')
figure.savefig(collage_path, dpi=300, bbox_inches='tight')
plt.show()

cutout_animation_path = visual_root / 'rubin_dp1_confirmed_lens_cutouts.gif'
cutout_labels = [
    f"{cutout['catalog']['main_id']} | {cutout['band']}-band"
    for cutout in cutouts
]
make_image_sequence_animation(
    [cutout['data'] for cutout in cutouts],
    cutout_labels,
    cutout_animation_path,
    duration_ms=800,
    image_size=640,
)
print(f'Animation contains {len(cutouts)} frames, one for every unique lens-band cutout.')
display(Image(filename=str(cutout_animation_path)))

## Pass every usable cutout to PCAT

DP1 coadds are background-subtracted and can contain negative pixels, so PCAT uses the calibrated variance plane through its Gaussian image likelihood. The fitted model has only three free parameters: Einstein radius and source-plane x and y positions. Source morphology, point-spread function, foreground light, and mass ellipticity remain fixed. These fits demonstrate data flow into PCAT and should not be interpreted as astrophysical lens models.

In [ ]:
fit_results = []
for cutout in cutouts:
    data = cutout['data']  # [nJy pixel^-1]
    variance = cutout['variance']  # [nJy^2 pixel^-2]
    if data.ndim != 2 or data.shape[0] != data.shape[1] or np.count_nonzero(np.isfinite(data)) < 100:
        print(f"Skipping unusable cutout {cutout['catalog']['main_id']} {cutout['band']}: shape={data.shape}")
        continue
    background = float(np.nanmedian(data))  # [nJy pixel^-1]
    source_flux = float(np.nansum(np.clip(data - background, 0.0, None)))  # [nJy]
    config = RomanLensConfig(
        number_side=data.shape[0],
        pixel_scale=0.2,  # [arcsec pixel^-1], fixed DP1 deep-coadd scale
        psf_fwhm=0.8,  # [arcsec], approximate fixed seeing for this demonstration
        background=background,  # [nJy pixel^-1]
        read_noise=0.0,
        source_counts=max(source_flux, 1.0),  # [nJy]
    )
    source_size = 0.35  # [arcsec]
    source_axis_ratio = 0.75
    source_angle = 0.0  # [rad]
    safe_name = re.sub(r'[^A-Za-z0-9]+', '_', str(cutout['catalog']['main_id'])).strip('_')
    posterior = pcat.main.sample(
        typeexpr='gener',
        retr_llik=gaussian_lens_log_likelihood,
        lens_config=config,
        lens_observed_image=data,
        lens_variance=variance,
        lens_source_size=source_size,
        lens_source_axis_ratio=source_axis_ratio,
        lens_source_angle=source_angle,
        parameter_names=('einstein_radius', 'source_x', 'source_y'),
        prior_types=('self', 'self', 'self'),
        prior_minima=(0.2, -2.0, -2.0),  # [arcsec]
        prior_maxima=(6.0, 2.0, 2.0),  # [arcsec]
        initial_values=(1.0, 0.0, 0.0),  # [arcsec]
        proposal_scales=(0.08, 0.03, 0.03),  # [arcsec]
        propwithsing=True,
        probpropblock=0.0,
        pathbase=str(output_root / 'pcat_runs'),
        strgcnfg=f'dp1_{safe_name}_{cutout["band"]}',
        numbproc=1,
        numbswep=300,
        numbburn=100,
        numbsamp=200,
        booladaptstdp=True,
        typeseed=42,
        typeverb=-1,
    )
    draws = np.asarray(posterior.listpostparagenrscalbase)
    median = np.median(draws, axis=0)  # [arcsec]
    model = render_lens_counts(config, median, source_size, source_axis_ratio, source_angle)
    fit_results.append({**cutout, 'config': config, 'draws': draws, 'median': median, 'model': model})
    print(f"{cutout['catalog']['main_id']} {cutout['band']}: median Einstein radius {median[0]:.2f} arcsec")

if len(fit_results) != len(cutouts):
    print(f'PCAT modeled {len(fit_results)} of {len(cutouts)} retrieved lens-band cutouts.')

In [ ]:
typefileplot = 'png'  # choose 'png' or 'pdf'
assert typefileplot in ('png', 'pdf')
for result in fit_results:
    data = result['data']  # [nJy pixel^-1]
    model = result['model']  # [nJy pixel^-1]
    residual_sigma = (data - model) / np.sqrt(result['variance'])
    limit = np.nanpercentile(data, 99.5)  # [nJy pixel^-1]
    figure, axes = plt.subplots(1, 3, figsize=(11, 3.5), constrained_layout=True)
    for axis, image, title, cmap, bounds, label in zip(
        axes,
        (data, model, residual_sigma),
        ('DP1 deep coadd', 'PCAT median model', 'Standardized residual'),
        ('Greys', 'Greys', 'RdBu_r'),
        ((np.nanpercentile(data, 1), limit), (np.nanpercentile(data, 1), limit), (-5, 5)),
        ('Flux [nJy pixel$^{-1}$]', 'Flux [nJy pixel$^{-1}$]', 'Residual [$\\sigma$]'),
    ):
        shown = axis.imshow(image, origin='lower', cmap=cmap, vmin=bounds[0], vmax=bounds[1], interpolation='nearest')
        axis.set_title(title)
        axis.set_xlabel('Pixel')
        axis.set_ylabel('Pixel')
        axis.grid(False)
        figure.colorbar(shown, ax=axis, label=label, shrink=0.8)
    main_id = str(result['catalog']['main_id'])
    figure.suptitle(f"{main_id}, {result['band']}-band; demonstration model")
    safe_name = re.sub(r'[^A-Za-z0-9]+', '_', main_id).strip('_')
    fit_path = visual_root / f'{safe_name}_{result["band"]}_pcat_fit.{typefileplot}'
    print(f'Writing to {fit_path}...')
    figure.savefig(fit_path, dpi=300, bbox_inches='tight')
    plt.show()
    plt.close(figure)

## Provenance and limits

- **Rubin data:** NSF–DOE Vera C. Rubin Observatory, Data Preview 1, DOI `10.71929/rubin/2570308`. DP1 uses real LSSTComCam observations but is available only to Rubin data-rights holders through the RSP.
- **Lens catalog:** SIMBAD `basic` rows with primary object type `gLS` or `gLe`, queried on the date printed above. Candidate types `LS?`, `Le?`, and `LI?` are excluded.
- **Completeness:** The collage and PCAT loop cover every unique lens-band DP1 deep-coadd overlap returned by that query. Duplicate coadd patches in one band are represented once. This does not establish that no uncataloged strong lenses exist in DP1.
- **Model:** Circular isothermal deflector, one fixed-shape Gaussian source, fixed approximate PSF, fixed source normalization, and independent Gaussian pixel errors. Real-lens inference requires foreground-light subtraction, an empirical coadd PSF, flexible source reconstruction, masks, and a more complete mass model.